### Google Colaboratory setup

In [ ]:
#@title 0. SQAIハンズオン環境のセットアップ

from pathlib import Path
import os
import subprocess
import sys


def running_on_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


ON_COLAB = running_on_colab()

if ON_COLAB:
    REPO_URL = "https://github.com/tiksato/SQAI_handson.git"
    REPO_REF = "main"
    PROJECT_ROOT = Path("/content/SQAI_handson")

    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--quiet",
                "--depth",
                "1",
                "--branch",
                REPO_REF,
                REPO_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

else:
    current = Path.cwd().resolve()

    if current.name == "SQAI_notebooks":
        PROJECT_ROOT = current.parent
    elif (current / "pyproject.toml").exists():
        PROJECT_ROOT = current
    else:
        raise RuntimeError(
            "SQAI_handsonのリポジトリルートを特定できません。"
        )

os.chdir(PROJECT_ROOT)

print("Environment:", "Google Colab" if ON_COLAB else "Local Python")
print("Project root:", PROJECT_ROOT)

In [ ]:
import time
import math
import numpy as np
from pyscf import gto, scf

from SQAI_modules.misc_utils.matrix_utilities import davidson
from SQAI_modules.ormas_tools.rasci import Full_CI, RAS_CI, RHF_CI

In [ ]:
basis_list = [
    'STO-3G', 
    '6-31G', 
    '6-31G(d,p)',
    '6-31G(df,pd)'
]

def calc_mo_integrals(mol):
    # Hamiltonian matrix elements generation                                                         
    print("Running RHF to obtain MO integrals...")
    t0 = time.time()
    HF = scf.RHF(mol)
    HF.verbose = 0
    HF.run()
    E_HF = HF.e_tot
    print(f"HF Energy:   {E_HF:.10f} Hartree")

    # Nuclear repulsion energy
    H_const = mol.energy_nuc()

    # AO integrals
    T = mol.intor("int1e_kin_sph") # AO kinetic
    Vnuc = mol.intor("int1e_nuc_sph") # AO Nuc-elec
    g_AO = mol.intor("int2e_sph").transpose(0,2,1,3) # AO ERIs in physics format

    # MO transformation
    CMO = HF.mo_coeff
    h_AO = T + Vnuc
    h_MO = np.einsum("pi,qj,pq->ij", CMO, CMO, h_AO, 
                     optimize=True)
    g_MO = np.einsum("pi,qj,rk,sl,pqrs->ijkl", CMO, CMO, CMO, CMO, g_AO, 
                     optimize=True)
    
    print(f" ... Done: {time.time()-t0:.2f} sec.")

    return H_const, h_MO, g_MO

def calc_fci(R_HH, basis, print_cic = False):

    print("##################################")
    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # Mol object generation                                                                         
    mol = gto.M(
        atom=f'''
        H   {-R_HH/2} 0 0
        H    {R_HH/2} 0 0
        ''',
        basis=basis, 
        symmetry=False, 
        verbose=0,
    )    
    n_act = mol.nao # number of active orbitals
    n_elec = np.array(mol.nelec) # active electrons, each spin                                                      
    print(f"basis = {basis}: n_act  = {n_act}")


    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # MO integrals generation                                                                         
    H_const, h_MO, g_MO = calc_mo_integrals(mol)


    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # CI instance generation                                                                         
    print("Constructing CI object...")
    t0 = time.time()
    myCI = Full_CI(
        n_elec = n_elec,
        n_orb = n_act, 
        num_threads = 10
    )
    print(f" ... Done: {time.time()-t0:.2f} sec.")
    print(f"CI dimension: {myCI.total_dim}")
    print("String distribution over occupation groups:")
    print(myCI.mat_num_str)


    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # Direct-CI Davidson diagonalization
    print("Davidson diagonalization...")
    t0 = time.time()
    def my_get_sigma(x):
        return myCI.h_prod(h_MO, g_MO, x).real

    hdiag = myCI.calc_hdiag(h_MO, g_MO).real
    def my_precond(dx, e, x0, small=1e-3):
        denom = hdiag - e
        return dx * denom / (denom**2 + small)

    Ene, CIC = davidson(myCI.total_dim,
                      my_get_sigma,
                      my_precond,
                      tol = 1E-6,
                      verbose=5)
    print(f" ... Done: {time.time()-t0:.2f} sec.")
    print(f"CI energy: {Ene + H_const}")    

    if print_cic:
        Ia = myCI.I_to_Ia
        Ib = myCI.I_to_Ib    
        for I,C in enumerate(CIC):
            astr = myCI.a_strings[Ia[I]].low
            bstr = myCI.b_strings[Ib[I]].low
            print(f"<{astr:0{n_act}b};{bstr:0{n_act}b}|Psi> = {C:+10.5f}")
    print()

    return Ene, CIC


Ene = {}
CIC = {}
for basis in basis_list:
    Ene[basis], CIC[basis] = calc_fci(R_HH = 1.0, 
                                      basis=basis, 
                                      print_cic=True)

In [ ]:
for R_HH in (1.0, 2.0, 4.0):
    print(f"R_HH = {R_HH}:")
    _, _CIC = calc_fci(R_HH = R_HH, basis='STO-3G', print_cic=True)
